# Training: nested context-state encoder, centralized critic, and both actors (CAGE-2)
All components are trained jointly from scratch under centralized training with decentralized execution, alternating a representation step on Eq. (3) (`Train_Encoder`) with a game step on the critic and both actors (`Train_Critic`, `Train_Player1`, `Train_Player2`).

Each episode draws a context, resets the simulator (with red's initial reconnaissance), and lets red play its head start of $h$ steps before 50 rounds of play. During training, blue's behavior policy is its actor with ε-greedy exploration (decaying from 1.0 to 0.1). Red plays a uniformly random action with probability ε (decaying from 0.3 to 0.1), and otherwise the most likely action of its network.

In [ ]:
%run Initialize.ipynb

## Settings

In [ ]:
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

GAMMA = 0.98
CRITIC_LR = 1e-5
PLAYER1_LR = 1e-5
PLAYER2_LR = 1e-5
ENCODER_LR = 5e-5
CRITIC_BATCH = 128
PLAYER1_BATCH = 128
PLAYER2_BATCH = 128
CONTEXT_BATCH = 128
TAU = 1e-4

# Eq. (3) weights, the Soft-InfoNCE temperature, and the kernel's locality scale
LAMBDA_R = 0.25
LAMBDA_P = 0.25
LAMBDA_THETA = 0.25
SI_TEMPERATURE = 0.5
SIGMA_THETA = 1.0

a_eps, a_eps_decay, a_eps_end = 1.0, 0.995, 0.1
u_eps, u_eps_decay, u_eps_end = 0.3, 0.99, 0.1

num_episodes = 10000
Update_Every = 10
UPDATE_START = 20

## Networks, optimizers, replay buffers

In [ ]:
# Representation: context encoder, its auxiliary heads, and EMA targets
Context_Model = TSTransformer().to(device)
trans_pred  = nn.Sequential(nn.Linear(Embedding_Size + n_a + real_state_dim, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                            nn.Linear(256, Embedding_Size)).to(device)
reward_head = nn.Sequential(nn.Linear(Embedding_Size, 128), nn.ReLU(), nn.Linear(128, 1)).to(device)
proj_online = nn.Sequential(nn.Linear(Embedding_Size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
pred_online = nn.Sequential(nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
target_model = copy.deepcopy(Context_Model)
proj_target  = copy.deepcopy(proj_online)
for p_ in list(target_model.parameters()) + list(proj_target.parameters()):
    p_.requires_grad_(False)
target_model.eval(); proj_target.eval()
encoder_optimizer = optim.AdamW(list(Context_Model.parameters()) + list(trans_pred.parameters()) +
                                list(reward_head.parameters()) + list(proj_online.parameters()) +
                                list(pred_online.parameters()), lr=ENCODER_LR, amsgrad=True)

# Game: centralized critic and both actors
critic = Critic().to(device)
critic_target = Critic().to(device)
critic_target.load_state_dict(critic.state_dict())
critic_optimizer = optim.AdamW(critic.parameters(), lr=CRITIC_LR, amsgrad=True)

Player1 = Player1Actor().to(device)
player1_optimizer = optim.AdamW(Player1.parameters(), lr=PLAYER1_LR, amsgrad=True)
Player2 = AttackerActor().to(device)
player2_optimizer = optim.AdamW(Player2.parameters(), lr=PLAYER2_LR, amsgrad=True)
red_act = red_policy(Player2)                      # used for red's head start

exp_memory = ReplayMemory(5000)
context_memory = ReplayMemory(500)
All_Loss = np.zeros([4, num_episodes])             # encoder, critic, blue, red
env_steps = 0

## Training

In [ ]:
T = lambda v, dtype=torch.float32: torch.tensor(v, dtype=dtype, device=device)

for i_episode in tqdm(range(num_episodes)):

    Theta, Type_Theta, w, p_exploit, h = Sample_Context()
    blue_state, blue_knowledge, red_state, red_knowledge, real_state = Reset_Episode()
    red_knowledge, red_state, real_state = head_start(red_act, h, red_knowledge, red_state, Type_Theta, p_exploit)
    Context = np.zeros([max_steps, CTX_DIM], dtype=np.float32)
    theta_Tensor, type_theta_Tensor = T(Theta), T(Type_Theta)

    for t in range(max_steps):
        mask = Context_Mask(t)
        Context_Tensor = T(Context).view(1, max_steps, CTX_DIM)
        x_Tensor, red_state_Tensor, blue_state_Tensor = T(real_state), T(red_state), T(blue_state)
        Context_Model.eval()
        with torch.no_grad():
            Embeding = Context_Model(Context_Tensor, mask).squeeze(0)
            a_dist = Player1(blue_state_Tensor, Embeding).cpu().numpy()          # Player1 already returns probabilities
            u_dist = Player2(red_state_Tensor, type_theta_Tensor).cpu().numpy()
        a_dist = a_dist / a_dist.sum()

        if np.random.rand() < a_eps:
            a = int(random.choices(range(n_a), weights=np.ones([n_a]) / n_a)[0])
        else:
            a = int(np.random.choice(n_a, p=a_dist))
        if np.random.rand() < u_eps:
            u = int(random.choices(range(n_u), weights=np.ones([n_u]) / n_u)[0])
        else:
            u = int(np.argmax(u_dist))

        _, red_knowledge, next_red_state, _ = red_step(u, red_knowledge, red_state, p_exploit)     # red moves first
        blue_action_name, y, next_real_state = blue_step(a)
        next_blue_state = blue_state_vector(y, blue_state, blue_action_name, t + 1)
        r = Reward(next_real_state, w, blue_action_name, y, blue_state)
        env_steps += 1

        Context[t, :] = Context_Row(blue_state, a, next_blue_state)

        if t > 0:
            next_mask = Context_Mask(t + 1)
            Next_Context_Tensor = T(Context).view(1, max_steps, CTX_DIM)
            a_Tensor, u_Tensor, Reward_Tensor = T(a, torch.long), T(u, torch.long), T(r)
            exp_memory.push([blue_state_Tensor.cpu(), T(next_blue_state).cpu(), x_Tensor.cpu(), T(next_real_state).cpu(),
                             red_state_Tensor.cpu(), T(next_red_state).cpu(), a_Tensor.cpu(), u_Tensor.cpu(),
                             Reward_Tensor.cpu(), type_theta_Tensor.cpu(), theta_Tensor.cpu(),
                             Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu()])
            context_memory.push([Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu(),
                                 a_Tensor.cpu(), T(y).cpu(), Reward_Tensor.cpu(), theta_Tensor.cpu()])

        if i_episode >= UPDATE_START and t % Update_Every == 0:
            All_Loss[0, i_episode] += Train_Encoder()          # representation step, Eq. (3)
            All_Loss[1, i_episode] += Train_Critic()           # game step
            All_Loss[2, i_episode] += Train_Player1()
            All_Loss[3, i_episode] += Train_Player2()

        blue_state, red_state, real_state = next_blue_state, next_red_state, next_real_state

    a_eps = max(a_eps_end, a_eps * a_eps_decay)
    u_eps = max(u_eps_end, u_eps * u_eps_decay)

## Save

In [ ]:
print(f"environment steps used in training (excluding head starts): {env_steps:,d}")
torch.save(Context_Model.state_dict(), 'encoder.pth')
torch.save(critic.state_dict(), 'critic.pth')
torch.save(Player1.state_dict(), 'player1.pth')
torch.save(Player2.state_dict(), 'player2.pth')